# m4 — Supervised ML Classification

> **⚠️ This module requires phenotypic resistance labels (R/S) linked to SRR accessions.**
> With small labeled N (<50), results should be treated as **exploratory**.
> Model performance will improve substantially with more labeled isolates.
> Module 4 can be run independently after Module 3 builds the feature table.

**Design follows the reference analysis** (`main_bothRandSremoved_Gscore_03022026.html`):

| Aspect | Detail |
|--------|--------|
| Outcome | Binary BDQ resistance: R=1, S=0; unknowns (U) excluded |
| Baseline features | `is_frameshift` + `dna_bind_category` (DNA-binding vs non-binding) |
| Enhanced features | + `grantham_score`, `mCSM_ddG_stability`, `MAESTRO_ddG`, `delta_KD_hydrophobicity`, `BLOSUM62`, `avg_pLDDT`, `pTM` |
| Models | Logistic Regression (L2), Random Forest (n=500), XGBoost |
| CV | Stratified 5-fold; G-score = primary metric |
| Class balance | `class_weight='balanced'` for LR/RF; `scale_pos_weight` for XGBoost |
| Outputs | Metrics CSV, ROC curves, feature importance, OOF probabilities |

In [ ]:
# parameters
DRIVE_OUTPUT   = "mmpR5_pipeline/output"
DRIVE_REF      = "mmpR5_pipeline/input"
PHENOTYPE_CSV  = ""    # optional override path for phenotype labels
MIN_LABELED    = 20    # minimum R+S rows to run ML

In [ ]:
# CPU only — no GPU needed
# ── Load pipeline config (Drive JSON fallback) ──────────────────────────────
import json, shutil, subprocess, warnings, time, datetime, concurrent.futures
from pathlib import Path
from Bio import SeqIO, Entrez
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import pandas as pd
import numpy as np
import requests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

_CFG_PATH = Path("/content/drive/MyDrive/ColabNotebooks/mmpR5_pipeline/pipeline_config.json")

def _load_config():
    if _CFG_PATH.exists():
        with open(_CFG_PATH) as _f:
            return json.load(_f)
    return {}

_cfg = _load_config()

def _p(key, default=None):
    """Resolve parameter: papermill-injected variable takes precedence over config JSON."""
    try:
        v = eval(key)                # injected by papermill
        return v if v is not None else _cfg.get(key, default)
    except Exception:
        return _cfg.get(key, default)

In [ ]:
# CPU only — no GPU needed
from google.colab import drive
drive.mount("/content/drive")
DRIVE_BASE  = Path("/content/drive/MyDrive/ColabNotebooks")
OUTPUT_ROOT = DRIVE_BASE / _p("DRIVE_OUTPUT", "mmpR5_pipeline/output")
REF_DIR     = DRIVE_BASE / _p("DRIVE_REF",    "mmpR5_pipeline/input")
MODULES_DIR = DRIVE_BASE / "mmpR5_pipeline" / "modules"
print(f"Drive mounted. Output root: {OUTPUT_ROOT}")

In [ ]:
# CPU only — no GPU needed
import os, warnings
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (roc_curve, auc, confusion_matrix,
                              accuracy_score, roc_auc_score)
from sklearn.inspection import permutation_importance as sk_perm
try:
    from xgboost import XGBClassifier
    _XGB_OK = True
except ImportError:
    _XGB_OK = False
    print("[WARN] xgboost not installed — XGBoost model skipped.")
warnings.filterwarnings("ignore")

_MIN_LABELED = int(_p("MIN_LABELED", 20))
_PHEN_CSV    = _p("PHENOTYPE_CSV", "")

# ── DNA-binding mutation catalogue (Lu et al. 2026) ───────────────────────────
_DNA_BINDING = {
    '132insG','132insGG','138insG','140insGG','141insG','192insG','211insG','234insG',
    'A59V','A62T','A62V','A84V','C46G','C46R','C46Y','E55D','G41V','G65R',
    'I67L','I67M','I67S','I67V','I80M','I80S','L40F','L40M','L40S','L40V',
    'L74M','L74V','L83F','L83P','M73I','N70D','P48H','P48L','Q51K','Q51R',
    'Q76E','R50Q','R72W','S52F','S52P','S53P','S63G','S63R','S68G','S68I',
    'S68N','S68R','T58P','V85A','W42C','W42R',
}

def _dna_bind_cat(hgvs_p):
    if pd.isna(hgvs_p) or str(hgvs_p).strip() in ("wild-type",""):
        return "WT"
    # Convert HGVS p. notation to short form (e.g. p.Ala59Val -> A59V)
    _h = str(hgvs_p).strip()
    # Short form check
    import re
    _m = re.match(r"p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2}|Ter)", _h)
    if _m:
        _aa1 = {"Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E",
                "Gly":"G","His":"H","Ile":"I","Leu":"L","Lys":"K","Met":"M","Phe":"F",
                "Pro":"P","Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V","Ter":"*"}
        wt3,pos,mut3 = _m.group(1),_m.group(2),_m.group(3)
        short = _aa1.get(wt3,wt3[0]) + pos + _aa1.get(mut3,mut3[0])
        return "DNA_binding" if short in _DNA_BINDING else "DNA_non_binding"
    # Indel
    if "indel" in _h.lower() or "ins" in _h.lower() or "del" in _h.lower() or "fs" in _h.lower():
        return "frameshift"
    return "unknown"

def g_score(y_true, y_pred):
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    sens = tp/(tp+fn) if (tp+fn)>0 else 0.0
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
    return float(np.sqrt(sens*spec)), sens, spec

print("ML module loaded. DNA-binding catalogue: %d mutations." % len(_DNA_BINDING))

In [ ]:
# CPU only — no GPU needed
# ── Load and prepare feature table ────────────────────────────────────────────
_ft_csv = OUTPUT_ROOT / "feature_table_all_samples.csv"
if not _ft_csv.exists():
    raise RuntimeError("feature_table_all_samples.csv not found. Run m3 first.")
_ft = pd.read_csv(str(_ft_csv))

# Merge external phenotype override if provided
if _PHEN_CSV and Path(_PHEN_CSV).exists():
    _phen_df = pd.read_csv(_PHEN_CSV)
    _ft = _ft.drop(columns=["phenotype"], errors="ignore")
    _ft = _ft.merge(_phen_df[["srr","phenotype"]], on="srr", how="left")
    _ft["phenotype"] = _ft["phenotype"].fillna("U")
    print(f"Phenotype override from {_PHEN_CSV}: {_ft['phenotype'].value_counts().to_dict()}")

# ── Feature engineering ───────────────────────────────────────────────────────
_ft["is_frameshift"] = (_ft["effect_type"] == "frameshift").astype(int)
_ft["dna_bind_category"] = _ft["Mutation (HGVS_p)"].apply(_dna_bind_cat)
_ft["dna_bind_enc"] = _ft["dna_bind_category"].map(
    {"DNA_binding":1,"DNA_non_binding":0,"frameshift":0,"WT":0,"unknown":0}).fillna(0)

# Rename for clarity
_col_map = {
    "Grantham_score":       "grantham_score",
    "BLOSUM62":             "blosum62",
    "delta_KD_hydrophobicity": "delta_kd_hydrophobicity",
    "mCSM_ddG_stability":   "mcsm_ddg_stability",
    "MAESTRO_ddG":          "maestro_ddg",
    "avg_pLDDT":            "avg_plddt",
    "pTM":                  "ptm",
    "delta_charge":         "delta_charge",
}
_ft = _ft.rename(columns=_col_map)

# Labeled subset
_labeled = _ft[_ft["phenotype"].isin(["R","S"])].copy()
_n_lab = len(_labeled); _n_r = (_labeled["phenotype"]=="R").sum(); _n_s = (_labeled["phenotype"]=="S").sum()

print(f"Total rows: {len(_ft)}")
print(f"Labeled (R+S): {_n_lab}  (R={_n_r}, S={_n_s})")
print(f"DNA-bind category distribution:\n{_ft['dna_bind_category'].value_counts().to_string()}")

if _n_lab < _MIN_LABELED:
    print()
    print("=" * 70)
    print(f"  [M4 SKIPPED] Labeled rows={_n_lab} < minimum={_MIN_LABELED}")
    print("  Provide phenotype labels (R/S) via PHENOTYPE_CSV or SRR_CSV.")
    print("=" * 70)
else:
    print(f"\nGate passed — proceeding with {_n_lab} labeled samples.")

In [ ]:
# CPU only — no GPU needed
# ── Model training and cross-validation ──────────────────────────────────────
if _n_lab >= _MIN_LABELED:
    # Baseline features: is_frameshift + dna_bind_enc
    _BASE_FEATS = ["is_frameshift","dna_bind_enc"]
    # Enhanced: add structural + physicochemical
    _STRUCT_FEATS = ["grantham_score","mcsm_ddg_stability","maestro_ddg",
                     "delta_kd_hydrophobicity","blosum62","delta_charge",
                     "avg_plddt","ptm"]
    _ALL_FEATS = _BASE_FEATS + _STRUCT_FEATS

    # Drop features with >50% missing in labeled set
    _miss50 = [c for c in _ALL_FEATS
               if c in _labeled.columns and _labeled[c].isna().mean() > 0.5]
    if _miss50: print(f"Dropping >50% missing: {_miss50}")
    _ENH_FEATS  = [c for c in _ALL_FEATS  if c not in _miss50 and c in _labeled.columns]
    _BASE_FEATS = [c for c in _BASE_FEATS if c not in _miss50 and c in _labeled.columns]

    _y = (_labeled["phenotype"]=="R").astype(int).values
    _cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    _models = {
        "LR_baseline": (
            Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),
                      ("clf",LogisticRegression(C=1.0,penalty="l2",class_weight="balanced",
                                                max_iter=1000,random_state=42))]),
            _BASE_FEATS),
        "LR_enhanced": (
            Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),
                      ("clf",LogisticRegression(C=1.0,penalty="l2",class_weight="balanced",
                                                max_iter=1000,random_state=42))]),
            _ENH_FEATS),
        "RF_baseline": (
            Pipeline([("imp",SimpleImputer(strategy="median")),
                      ("clf",RandomForestClassifier(n_estimators=500,class_weight="balanced",
                                                    random_state=42))]),
            _BASE_FEATS),
        "RF_enhanced": (
            Pipeline([("imp",SimpleImputer(strategy="median")),
                      ("clf",RandomForestClassifier(n_estimators=500,class_weight="balanced",
                                                    random_state=42))]),
            _ENH_FEATS),
    }
    if _XGB_OK:
        _spw = max(_n_s,1)/max(_n_r,1)
        _models["XGB_enhanced"] = (
            Pipeline([("imp",SimpleImputer(strategy="median")),
                      ("clf",XGBClassifier(n_estimators=200,scale_pos_weight=_spw,
                                           eval_metric="logloss",random_state=42,verbosity=0))]),
            _ENH_FEATS)

    _results = {}; _oof_proba = {}

    for _mname, (_pipe, _feats) in _models.items():
        _X = _labeled[_feats].values
        _folds = {"auc":[],"acc":[],"sens":[],"spec":[],"gscore":[]}
        _oof = np.zeros(len(_y))
        print(f"  {_mname} (feats={len(_feats)}) ...", end=" ")
        for _tr_i, _val_i in _cv.split(_X, _y):
            _Xtr,_Xvl = _X[_tr_i],_X[_val_i]; _ytr,_yvl = _y[_tr_i],_y[_val_i]
            if "XGB" in _mname:
                _sw = np.where(_ytr==1, max(_n_s,1)/max(_n_r,1), 1.0)
                _pipe.fit(_Xtr,_ytr,clf__sample_weight=_sw)
            else:
                _pipe.fit(_Xtr,_ytr)
            _ypr = _pipe.predict_proba(_Xvl)[:,1]; _oof[_val_i] = _ypr
            _yp  = _pipe.predict(_Xvl)
            _gs,_se,_sp = g_score(_yvl,_yp)
            _folds["auc"].append(roc_auc_score(_yvl,_ypr) if len(np.unique(_yvl))>1 else float("nan"))
            _folds["acc"].append(accuracy_score(_yvl,_yp))
            _folds["sens"].append(_se); _folds["spec"].append(_sp); _folds["gscore"].append(_gs)
        _mn = {k:float(np.nanmean(v)) for k,v in _folds.items()}
        _sd = {k:float(np.nanstd(v))  for k,v in _folds.items()}
        _results[_mname] = {"means":_mn,"sds":_sd,"feats":_feats,"pipe":_pipe}
        _oof_proba[_mname] = _oof
        print(f"G={_mn['gscore']:.3f}±{_sd['gscore']:.3f}  AUC={_mn['auc']:.3f}")

    # ── Metrics table ─────────────────────────────────────────────────────────
    _met_rows = []
    for _mn2,_r in _results.items():
        _met_rows.append({"model":_mn2,
            "AUC":f"{_r['means']['auc']:.3f}±{_r['sds']['auc']:.3f}",
            "Accuracy":f"{_r['means']['acc']:.3f}±{_r['sds']['acc']:.3f}",
            "Sensitivity":f"{_r['means']['sens']:.3f}±{_r['sds']['sens']:.3f}",
            "Specificity":f"{_r['means']['spec']:.3f}±{_r['sds']['spec']:.3f}",
            "G_score":f"{_r['means']['gscore']:.3f}±{_r['sds']['gscore']:.3f}"})
    _met_df = pd.DataFrame(_met_rows)
    print("\n" + "="*70 + "\n  5-FOLD CV METRICS (primary: G-score)\n" + "="*70)
    print(_met_df.to_string(index=False))

    # Save metrics
    _ml_dir = OUTPUT_ROOT / "07_ml_results"; _ml_dir.mkdir(parents=True, exist_ok=True)
    _met_df.to_csv(str(_ml_dir/"ml_cv_metrics.csv"), index=False)

    # ── ROC curves ────────────────────────────────────────────────────────────
    _palette = {"LR_baseline":"#adb5bd","LR_enhanced":"#003f88",
                "RF_baseline":"#90e0ef","RF_enhanced":"#00b4d8","XGB_enhanced":"#e63946"}
    _fig, _ax = plt.subplots(figsize=(7,6))
    for _mn3,_proba in _oof_proba.items():
        if len(np.unique(_y)) > 1:
            _fpr,_tpr,_ = roc_curve(_y,_proba)
            _ax.plot(_fpr,_tpr,color=_palette.get(_mn3,"gray"),
                     label=f"{_mn3} (AUC={auc(_fpr,_tpr):.3f})",
                     lw=2,ls="--" if "baseline" in _mn3 else "-")
    _ax.plot([0,1],[0,1],"k--",lw=1)
    _ax.set(xlabel="FPR",ylabel="TPR",title="ROC — BDQ resistance (5-fold OOF)")
    _ax.legend(loc="lower right",fontsize=8); plt.tight_layout()
    _roc_png = _ml_dir/"roc_curves.png"
    plt.savefig(str(_roc_png),dpi=150); plt.show(); plt.close()

    # ── Feature importance (best enhanced model by G-score) ───────────────────
    _best_key = max((k for k in _results if "enhanced" in k),
                    key=lambda k: _results[k]["means"]["gscore"])
    _best_pipe = _results[_best_key]["pipe"]; _best_feats = _results[_best_key]["feats"]
    _best_pipe.fit(_labeled[_best_feats].values, _y)
    if "LR" in _best_key:
        _imp_m = np.abs(_best_pipe.named_steps["clf"].coef_[0]); _imp_s = np.zeros_like(_imp_m)
        _imp_lbl = "|Coefficient|"
    else:
        _perm = sk_perm(_best_pipe,_labeled[_best_feats].values,_y,n_repeats=20,random_state=42)
        _imp_m = _perm.importances_mean; _imp_s = _perm.importances_std; _imp_lbl = "Permutation importance"
    _imp_df = (pd.DataFrame({"feature":_best_feats,"importance":_imp_m,"std":_imp_s})
               .sort_values("importance",ascending=False))
    _fig2, _ax2 = plt.subplots(figsize=(8,5))
    _ax2.barh(_imp_df["feature"][::-1],_imp_df["importance"][::-1],
              xerr=_imp_df["std"][::-1],color="#003f88",alpha=0.8)
    _ax2.set(xlabel=_imp_lbl,title=f"Feature importance — {_best_key}")
    plt.tight_layout()
    _fi_png = _ml_dir/"feature_importance.png"; plt.savefig(str(_fi_png),dpi=150); plt.show(); plt.close()

    # ── Save OOF probabilities to feature table ───────────────────────────────
    _ft_out = _ft.copy(); _ft_out["ml_pred_proba_R"] = float("nan")
    _lab_idx = _ft[_ft["phenotype"].isin(["R","S"])].index
    _ft_out.loc[_lab_idx,"ml_pred_proba_R"] = _oof_proba[_best_key]
    _ft_out.to_csv(str(OUTPUT_ROOT/"feature_table_all_samples_with_ml.csv"),index=False)
    for _f in [_ml_dir/"ml_cv_metrics.csv",_roc_png,_fi_png]:
        if Path(_f).exists(): shutil.copy2(str(_f),str(_ml_dir/_f.name))
    print(f"\nBest enhanced model: {_best_key} (G-score={_results[_best_key]['means']['gscore']:.3f})")
    print(f"ML results saved to {_ml_dir}")
    display(_met_df)